[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/laleye/IA-Training-participants/blob/main/module1/session2/tp/TP2/TP2_analyse_nettoyage.ipynb)

# TP2 — Analyse de spectrogrammes & nettoyage de mon corpus

**Travail inter-session après la session 2** · ≈ 2 h 30 · Ticket `M1-S2-07`

Ce notebook accompagne `README.md`. Quatre parties + un bonus :
1. analyse acoustique comparée de 3 paires minimales (vos voix + le pack) ;
2. nettoyage de tous vos fichiers TP1 avec votre pipeline (paramètres justifiés) ;
3. segmentation de l'enregistrement long fourni ;
4. rapport qualité ;
5. 🔴 bonus : MFCC *from scratch*.

In [ ]:
# ⏳ Installation (≈ 1 min sur Colab, instantané ensuite). Ne pas modifier.
import importlib.util, subprocess, sys
if importlib.util.find_spec("m1utils") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "m1utils[all] @ git+https://github.com/laleye/IA-Training-participants.git"], check=True)
import m1utils
INFO = m1utils.setup_notebook()

In [ ]:
# 📝 PARAMÈTRES
LANGUE = "fon"
DOSSIER_TP1 = "/content/drive/MyDrive/TP1_nom"   # entrée : vos enregistrements
DOSSIER_TP2 = "/content/drive/MyDrive/TP2_nom"   # sortie : clean/, segments_long/, rapport_qualite.csv

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from m1utils.audio import load_audio, play, save_audio
from m1utils.data import load_manifest
from m1utils.features import f0_contour, mfcc
from m1utils.preprocess import (dbfs, detect_clipping, estimate_snr, normalize_loudness, normalize_peak, remove_dc,
                                split_on_segments, trim_silence, vad_segments)
from m1utils.viz import plot_f0, plot_spectrogram

if INFO["colab"] and DOSSIER_TP1.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

entree = Path(DOSSIER_TP1) / "audio"
MODE_EXEMPLE = not entree.exists()
if MODE_EXEMPLE:
    display(Markdown("⚠️ **Mode exemple** : vos enregistrements TP1 sont introuvables ; on utilise les packs du cours."))
    fichiers = [Path(p) for p in load_manifest("demo_s1").abspath]
    sortie = Path("/tmp/TP2_exemple")
else:
    fichiers = sorted(p for p in entree.iterdir() if p.suffix.lower() in {".wav", ".flac", ".m4a", ".mp3"})
    sortie = Path(DOSSIER_TP2)
sortie.mkdir(parents=True, exist_ok=True)
print(f"{len(fichiers)} fichiers → sorties dans {sortie}")

## 1. Analyse comparée de 3 paires minimales 🟢
Pour chaque paire : spectrogrammes + F0 côte à côte, pour **votre** voix et pour le pack du cours.
Renseignez les identifiants de vos fichiers (TP1).

In [ ]:
paires_pack = load_manifest("minimal_pairs")
if MODE_EXEMPLE:
    MES_PAIRES = []
else:
    MES_PAIRES = [("fon_spk01_fon_01", "fon_spk01_fon_02"),   # ✍️ vos identifiants
                  ("fon_spk01_fon_05", "fon_spk01_fon_06"),
                  ("fon_spk01_fon_07", "fon_spk01_fon_08")]
chemins = {f.stem: f for f in fichiers}

def afficher_paire(items, titre):
    fig, axes = plt.subplots(1, len(items), figsize=(7 * len(items), 3.6))
    for ax, (etiquette, chemin) in zip(np.atleast_1d(axes), items):
        y, sr = load_audio(chemin)
        plot_f0(y, sr, ax=ax, title=etiquette)
    fig.suptitle(titre)
    plt.tight_layout()
    plt.show()

for a, b in MES_PAIRES:
    afficher_paire([(a, chemins[a]), (b, chemins[b])], "Ma voix")
for paire, groupe in paires_pack[paires_pack.lang == LANGUE].groupby("pair"):
    afficher_paire(list(zip(groupe.text, groupe.abspath)), f"Pack du cours — paire {paire} ({groupe.contrast.iloc[0]})")

**✍️ Interprétation (5 à 8 lignes par paire)** : quel indice acoustique porte le contraste (F0, durée, formants) ?
Le contraste est-il plus net chez vous ou dans la voix synthétique ? Quelles limites de mesure ?

✍️ *Votre analyse :* …

## 2. Nettoyer tous mes fichiers 🔵
Appliquez **votre** pipeline (celui de S2-NB02, ou `m1utils.preprocess.preprocess`) et **justifiez** chaque paramètre
dans la cellule markdown suivante.

In [ ]:
PARAMS = {"normalisation": "peak", "peak_dbfs": -1.0, "top_db": 30, "marge_s": 0.2}   # ✍️ à justifier

def mon_pipeline(chemin, params=PARAMS):
    ...   # ✍️ À COMPLÉTER : charger 16 kHz mono → retirer DC → normaliser → trim → marge ; renvoyer (y, sr)


lignes = []
for f in fichiers:
    brut, sr0 = load_audio(f, sr=None)
    propre, sr = mon_pipeline(f)
    chemin = save_audio(sortie / "clean" / f"{f.stem}.wav", propre, sr)
    lignes.append({"id": f.stem, "path": str(chemin), "duration_s": round(len(propre) / sr, 3), "sr": sr,
                   "snr_avant_db": round(estimate_snr(brut, sr0), 1), "snr_db": round(estimate_snr(propre, sr), 1),
                   "rms_avant_dbfs": round(dbfs(brut, "rms"), 1), "rms_apres_dbfs": round(dbfs(propre, "rms"), 1),
                   "clipping_ratio": round(detect_clipping(brut), 4)})
rapport = pd.DataFrame(lignes)
display(rapport)

**✍️ Justification des paramètres** (normalisation, `top_db`, débruitage oui/non et pourquoi — appuyez-vous sur la
mesure de S2-NB02) : …

## 3. Segmenter l'enregistrement long 🔵
Découpez l'émission radio wolof fournie (168 s) en énoncés de 2 à 15 s ; visez ≥ 80 % d'énoncés dans cette plage.

In [ ]:
long = load_manifest("long_recording_s2")
emission, sr = load_audio(long.abspath.iloc[0])
emission = normalize_peak(emission)
MIN_SILENCE_S = ...   # ✍️ À COMPLÉTER (réglage justifié)
segments = ...        # ✍️ À COMPLÉTER : vad_segments(...)
morceaux = split_on_segments(emission, segments, sr, pad_s=0.1, max_segment_s=15)
durees = np.array([len(m) / sr for m in morceaux])
for k, m in enumerate(morceaux):
    save_audio(sortie / "segments_long" / f"long_{k:03d}.wav", m, sr)
part = np.mean((durees >= 2) & (durees <= 15))
print(f"{len(morceaux)} énoncés · {part:.0%} dans [2, 15] s · médiane {np.median(durees):.1f} s")
plt.hist(durees, bins=np.arange(0, 16, 1))
plt.xlabel("durée (s)")
plt.title("Durées des énoncés segmentés")
plt.show()

## 4. Rapport qualité 🟢
Le tableau ci-dessous est votre `rapport_qualite.csv` (entrée de la session 3). Complétez la colonne **décision**
(`garder` / `rejeter` / `réenregistrer`) selon des règles explicites, puis rédigez 3 recommandations.

In [ ]:
def decision(r):
    if r.clipping_ratio > 0.001:
        return "réenregistrer"          # saturation : irréversible
    if r.snr_db < 10 or r.duration_s < 0.5:
        return "rejeter"
    return "garder"

rapport["décision"] = rapport.apply(decision, axis=1)
rapport.to_csv(sortie / "rapport_qualite.csv", index=False)
display(rapport["décision"].value_counts())
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(rapport.snr_avant_db, rapport.snr_db)
ax.plot([0, 40], [0, 40], "k--", lw=1)
ax.set(xlabel="SNR avant (dB)", ylabel="SNR après (dB)", title="Effet du pipeline sur le SNR estimé")
plt.show()

**✍️ Trois recommandations pour les prochaines collectes** (matériel, environnement, protocole) : …

## 5. Bonus 🔴 — MFCC *from scratch*
Implémentez les MFCC en NumPy et comparez à `librosa` (même paramètres : 25 ms / 10 ms, 40 filtres Mel, 13 coefficients).
Étapes : (1) pré-accentuation $y[n] - 0{,}97\,y[n-1]$ (optionnelle ; désactivée ici pour comparer à librosa) ;
(2) découpage en trames + fenêtre de Hann ; (3) |FFT|² ; (4) banc de filtres Mel ; (5) log ; (6) DCT de type II.

In [ ]:
...  # ✍️ À COMPLÉTER

## ✅ Avant de rendre
- [ ] Notebook exécuté sur **vos** données (pas en mode exemple), parties 1 à 4 remplies.
- [ ] `clean/`, `segments_long/` et `rapport_qualite.csv` présents dans `TP2_{nom}/`.
- [ ] Paramètres justifiés, 3 recommandations rédigées.

Rendu : dossier Drive `TP2_{nom}/` partagé avec le formateur, **veille de la session 3, 18 h**.